# 01 — Ingesta RAW de NYC Taxi Data a Snowflake

Este notebook descarga archivos Parquet (yellow + green) desde la fuente pública y los carga en `NYC_TAXI_P3.RAW.TRIPS_RAW`.
También carga la tabla de zonas (`TAXI_ZONE_LOOKUP`) y registra auditoría en `INGESTION_LOG`.

In [ ]:
import os
import uuid
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# JARs de Snowflake ya están en /usr/local/spark/jars/ (instalados por Dockerfile)
spark = (SparkSession.builder
    .appName("NYC_Taxi_Ingestion")
    .getOrCreate())

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}

PARQUET_BASE = os.environ["PARQUET_BASE_URL"]
YEAR_START   = int(os.environ["YEAR_START"])
YEAR_END     = int(os.environ["YEAR_END"])
MONTH_START  = int(os.environ["MONTH_START"])
MONTH_END    = int(os.environ["MONTH_END"])
SERVICES     = os.environ["SERVICES"].split(",")
RUN_ID       = os.environ.get("RUN_ID", str(uuid.uuid4())[:8])

print(f"RUN_ID: {RUN_ID}")
print(f"Range: {YEAR_START}-{MONTH_START:02d} to {YEAR_END}-{MONTH_END:02d}")
print(f"Services: {SERVICES}")

## Taxi Zone Lookup ingestion

In [ ]:
zone_url = os.environ["TAXI_ZONE_URL"]
pdf_zones = __import__("pandas").read_csv(zone_url)
df_zones = spark.createDataFrame(pdf_zones)

(df_zones.write
    .format("snowflake")
    .options(**SF_OPTIONS)
    .option("dbtable", "TAXI_ZONE_LOOKUP")
    .mode("overwrite")
    .save())

print(f"Taxi zones loaded: {df_zones.count()} rows")

## Monthly ingestion function (idempotent)

In [ ]:
def generate_months(y_start, y_end, m_start, m_end):
    months = []
    for year in range(y_start, y_end + 1):
        m_lo = m_start if year == y_start else 1
        m_hi = m_end   if year == y_end   else 12
        for month in range(m_lo, m_hi + 1):
            months.append((year, month))
    return months

YELLOW_COLS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge", "Airport_fee"
]

GREEN_COLS = [
    "VendorID", "lpep_pickup_datetime", "lpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "trip_type", "ehail_fee"
]

ALL_COLS = [
    "VendorID",
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "lpep_pickup_datetime", "lpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "trip_type", "ehail_fee",
    "run_id", "service_type", "source_year", "source_month",
    "ingested_at_utc", "source_path"
]

## Ingest one month (idempotent via DELETE + INSERT)

In [ ]:
import requests

def ingest_one_month(service, year, month, run_id):
    source_month_str = f"{year}-{month:02d}"
    url = f"{PARQUET_BASE}/{service}_tripdata_{source_month_str}.parquet"
    local_path = f"/home/jovyan/data/{service}_{source_month_str}.parquet"
    started = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    
    try:
        resp = requests.head(url, timeout=30)
        if resp.status_code in (403, 404):
            print(f"[SKIP] {service} {source_month_str} not available (HTTP {resp.status_code})")
            return {"service": service, "year": year, "month": month, "status": "skipped", "rows": 0}
    except Exception as e:
        print(f"[ERROR] {service} {source_month_str}: {e}")
        return {"service": service, "year": year, "month": month, "status": "error", "rows": 0}

    print(f"[START] {service} {source_month_str} — downloading...")
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    with open(local_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print(f"[DOWNLOADED] {service} {source_month_str}")

    df = spark.read.parquet(local_path)
    total_rows = df.count()

    for col in ALL_COLS:
        if col not in df.columns:
            df = df.withColumn(col, F.lit(None))

    now_ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    df = (df
        .withColumn("run_id",          F.lit(run_id))
        .withColumn("service_type",    F.lit(service))
        .withColumn("source_year",     F.lit(year))
        .withColumn("source_month",    F.lit(month))
        .withColumn("ingested_at_utc", F.to_timestamp(F.lit(now_ts)))
        .withColumn("source_path",     F.lit(url))
        .select(ALL_COLS)
    )

    # IDEMPOTENCY: delete existing data for this month/service before writing
    delete_sql = f"""
        DELETE FROM NYC_TAXI_P3.RAW.TRIPS_RAW
        WHERE service_type = '{service}'
          AND source_year  = {year}
          AND source_month = {month}
    """
    spark._jvm.net.snowflake.spark.snowflake.Utils.runQuery(SF_OPTIONS, delete_sql)
    print(f"[IDEMPOTENT] Deleted existing rows for {service} {source_month_str}")

    (df.write
        .format("snowflake")
        .options(**SF_OPTIONS)
        .option("dbtable", "TRIPS_RAW")
        .mode("append")
        .save())

    finished = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    log_df = spark.createDataFrame([{
        "run_id": run_id, "service_type": service,
        "source_year": year, "source_month": month,
        "source_path": url, "rows_loaded": total_rows,
        "rows_in_source": total_rows, "status": "loaded",
        "error_message": None, "started_at_utc": started,
        "finished_at_utc": finished
    }])
    (log_df.write
        .format("snowflake")
        .options(**SF_OPTIONS)
        .option("dbtable", "INGESTION_LOG")
        .mode("append")
        .save())

    import os as _os
    _os.remove(local_path)
    print(f"[DONE] {service} {source_month_str}: {total_rows} rows loaded")
    return {"service": service, "year": year, "month": month, "status": "loaded", "rows": total_rows}

## Run the full backfill

In [ ]:
months = generate_months(YEAR_START, YEAR_END, MONTH_START, MONTH_END)
results = []

for service in SERVICES:
    for year, month in months:
        result = ingest_one_month(service, year, month, RUN_ID)
        results.append(result)

import pandas as pd
df_results = pd.DataFrame(results)
print(f"\n=== INGESTION SUMMARY ===")
print(f"Total months attempted: {len(df_results)}")
print(f"Loaded:  {len(df_results[df_results['status']=='loaded'])}")
print(f"Skipped: {len(df_results[df_results['status']=='skipped'])}")
print(f"Errors:  {len(df_results[df_results['status']=='error'])}")
df_results